In [1]:
!pwd

/content


In [2]:
import sys;
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
# !pip install uv

# !uv pip install \
#   "darts==0.34.0" \
#   "torch==2.3.1" \
#   "pytorch-lightning==2.2.5" \
#   "torchmetrics==1.3.2" \
#   "numpy==1.26.4" \
#   "pandas==2.2.2" \
#   "scikit-learn==1.7.2" \
#   "dask==2025.5.0" \
#   "xgboost==3.0.2" \
#   "catboost==1.2.8" \
#   "pyyaml==6.0.2" \
#   "lightgbm==4.6.0"

# Pin torch for reproducibility with your local run
# !uv pip install -q --index-url https://download.pytorch.org/whl/cu121 "torch==2.3.1"

In [4]:
import sys, torch, darts, pytorch_lightning as pl, numpy as np, pandas as pd
print(sys.version)
print("torch:", torch.__version__)
print("darts:", darts.__version__)
print("lightning:", pl.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("cuda:", torch.cuda.is_available())

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.3.1+cu121
darts: 0.34.0
lightning: 2.2.5
numpy: 1.26.4
pandas: 2.2.2
cuda: False


**Forecasts**
 - **Country: Canada**
 - **Forecast Horizons:12M and 24M Forecast**

In [5]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import LightGBMModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class canadaLGBMForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the LGBM forecast generator for canada data

      Args:
          data_path: Path to the canada data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> LightGBMModel:
      """
      Create and return the LGBM model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured LightGBMModel
      """
      return LightGBMModel(
          lags=forecast_horizon,
          lags_past_covariates=forecast_horizon,
          output_chunk_length=forecast_horizon,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              # verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(n=horizon)

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'canada_lgbm_forecasts_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the LGBM forecast generation
  """
  # Initialize forecast generator
  generator = canadaLGBMForecastGenerator(
      data_path='all_mulvar_data_canada_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"canada_lgbm_forecasts_{horizon}m.csv")

if __name__ == "__main__":
  main()

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Forecast saved to canada_lgbm_forecasts_24m.csv

Created files:
canada_lgbm_forecasts_12m.csv
canada_lgbm_forecasts_24m.csv


**Forecasts**
 - **Country: USA**
 - **Forecast Horizons:12M and 24M Forecast**

In [6]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import LightGBMModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class USALGBMForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the LGBM forecast generator for USA data

      Args:
          data_path: Path to the USA data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> LightGBMModel:
      """
      Create and return the LGBM model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured LightGBMModel
      """
      return LightGBMModel(
          lags=forecast_horizon,
          lags_past_covariates=forecast_horizon,
          output_chunk_length=forecast_horizon,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              # verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(n=horizon)

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'usa_lgbm_forecasts_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the LGBM forecast generation
  """
  # Initialize forecast generator
  generator = USALGBMForecastGenerator(
      data_path='all_mulvar_data_usa_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"usa_lgbm_forecasts_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# usa_lgbm_forecasts_12m.csv
# usa_lgbm_forecasts_24m.csv

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Forecast saved to usa_lgbm_forecasts_24m.csv

Created files:
usa_lgbm_forecasts_12m.csv
usa_lgbm_forecasts_24m.csv


**Forecasts**
 - **Country: France**
 - **Forecast Horizons:12M and 24M Forecast**

In [7]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import LightGBMModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class franceLGBMForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the LGBM forecast generator for france data

      Args:
          data_path: Path to the france data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> LightGBMModel:
      """
      Create and return the LGBM model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured LightGBMModel
      """
      return LightGBMModel(
          lags=forecast_horizon,
          lags_past_covariates=forecast_horizon,
          output_chunk_length=forecast_horizon,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              # verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(n=horizon)

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'france_lgbm_forecasts_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the LGBM forecast generation
  """
  # Initialize forecast generator
  generator = franceLGBMForecastGenerator(
      data_path='all_mulvar_data_france_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"france_lgbm_forecasts_{horizon}m.csv")

if __name__ == "__main__":
  main()

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000640 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9761
[LightGBM] [Info] Number of data points in the train set: 316, number of used features: 108
[LightGBM] [Info] Start training from score 104.317595
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

**Forecasts**
 - **Country: Germany**
 - **Forecast Horizons:12M and 24M Forecast**

In [8]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import LightGBMModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class germanyLGBMForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the LGBM forecast generator for germany data

      Args:
          data_path: Path to the germany data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> LightGBMModel:
      """
      Create and return the LGBM model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured LightGBMModel
      """
      return LightGBMModel(
          lags=forecast_horizon,
          lags_past_covariates=forecast_horizon,
          output_chunk_length=forecast_horizon,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              # verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(n=horizon)

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'germany_lgbm_forecasts_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the LGBM forecast generation
  """
  # Initialize forecast generator
  generator = germanyLGBMForecastGenerator(
      data_path='all_mulvar_data_germany_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"germany_lgbm_forecasts_{horizon}m.csv")

if __name__ == "__main__":
  main()

# Created/Modified files during execution:
# germany_lgbm_forecasts_12m.csv
# germany_lgbm_forecasts_24m.csv

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Forecast saved to germany_lgbm_forecasts_24m.csv

Created files:
germany_lgbm_forecasts_12m.csv
germany_lgbm_forecasts_24m.csv


**Forecasts**
 - **Country: Japan**
 - **Forecast Horizons:12M and 24M Forecast**

In [9]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import LightGBMModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class ukLGBMForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the LGBM forecast generator for uk data

      Args:
          data_path: Path to the uk data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> LightGBMModel:
      """
      Create and return the LGBM model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured LightGBMModel
      """
      return LightGBMModel(
          lags=forecast_horizon,
          lags_past_covariates=forecast_horizon,
          output_chunk_length=forecast_horizon,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              # verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(n=horizon)

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'uk_lgbm_forecasts_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the LGBM forecast generation
  """
  # Initialize forecast generator
  generator = ukLGBMForecastGenerator(
      data_path='all_mulvar_data_uk_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"uk_lgbm_forecasts_{horizon}m.csv")

if __name__ == "__main__":
  main()

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Forecast saved to uk_lgbm_forecasts_24m.csv

Created files:
uk_lgbm_forecasts_12m.csv
uk_lgbm_forecasts_24m.csv


**Forecasts**
 - **Country: Italy**
 - **Forecast Horizons:12M and 24M Forecast**

In [10]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import LightGBMModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class italyLGBMForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      """
      Initialize the LGBM forecast generator for italy data

      Args:
          data_path: Path to the italy data CSV file
          random_seed: Random seed for reproducibility
      """
      # Set random seeds
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      """
      Prepare training data based on forecast horizon

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Tuple of target and exogenous TimeSeries objects
      """
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> LightGBMModel:
      """
      Create and return the LGBM model with horizon-specific parameters

      Args:
          forecast_horizon: Number of months to forecast (12 or 24)

      Returns:
          Configured LightGBMModel
      """
      return LightGBMModel(
          lags=forecast_horizon,
          lags_past_covariates=forecast_horizon,
          output_chunk_length=forecast_horizon,
          random_state=0
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      """
      Generate forecasts for specified horizons

      Args:
          forecast_horizons: List of forecast horizons (e.g., [12, 24])

      Returns:
          Dictionary with forecast horizons as keys and forecast DataFrames as values
      """
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              # verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(n=horizon)

          # Convert to DataFrame
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'italy_lgbm_forecasts_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  """
  Main function to run the LGBM forecast generation
  """
  # Initialize forecast generator
  generator = italyLGBMForecastGenerator(
      data_path='all_mulvar_data_italy_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"italy_lgbm_forecasts_{horizon}m.csv")

if __name__ == "__main__":
  main()

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Forecast saved to italy_lgbm_forecasts_24m.csv

Created files:
italy_lgbm_forecasts_12m.csv
italy_lgbm_forecasts_24m.csv
